In [ ]:
%pip install -q pinecone langchain-text-splitters langchain-community sentence-transformers python-dotenv

# Week 5 Wednesday -- Pinecone Operations, Text Splitting & Semantic Search

On Tuesday we got Pinecone set up, created our first index, and performed basic upsert/query operations. Today we go deeper: we'll explore **production text splitting strategies**, learn how to **load and process documents** from various formats, attach **rich metadata** to our vectors, and run **filtered semantic searches** against Pinecone — all skills you'll need for a real RAG pipeline.

## Learning Objectives

By the end of this notebook you will be able to:

1. Apply multiple text splitting strategies (recursive, sentence-based, semantic) to prepare documents for embedding.
2. Use LangChain document loaders to ingest Markdown, CSV, JSON, and other formats.
3. Design metadata schemas and attach structured metadata to Pinecone vectors.
4. Execute semantic search queries with **metadata filters** in Pinecone.
5. Interpret similarity scores and recognize vector search limitations.
6. Understand when and why to consider **hybrid search** approaches.

## Roadmap

| Stage | Topic | Key Concepts |
|-------|-------|--------------|
| **1** | Text Splitting Strategies | `RecursiveCharacterTextSplitter`, Markdown/code splitting, chunk overlap, sentence-based splitting |
| **2** | Document Loaders & Metadata | LangChain loaders (CSV, JSON, Web), metadata schema design, upsert with metadata into Pinecone |
| **3** | Semantic Search with Pinecone | Querying with filters, score interpretation, search limitations, hybrid search concepts |

---

# Stage 1 — Text Splitting Strategies

Before any document enters a vector database it must be **chunked** into pieces small enough to embed. The splitter you choose determines whether a query like *"What is machine learning?"* returns a coherent explanation — or a mid-sentence fragment like *"earning rate of 0.01 is typically..."*.

LangChain ships production-ready splitters so you don't have to build your own. The main families are:

| Family | How It Works | Best For |
|--------|-------------|----------|
| **Character-based** | Fixed character count | Simple, predictable chunks |
| **Recursive** | Tries multiple separators in order | Most documents — the go-to default |
| **Token-based** | Splits by token count | When LLM token limits matter |
| **Semantic / Sentence** | Splits at sentence boundaries | Preserving meaning across chunks |

We'll focus on the **RecursiveCharacterTextSplitter** (the workhorse) and then look at Markdown-aware and code-aware variants.

## 1.1 — RecursiveCharacterTextSplitter

This splitter tries a list of separators **in order of preference** (`"\n\n"`, `"\n"`, `" "`, `""`) and falls back through the list when a chunk is still too large. The result is chunks that respect paragraph and sentence boundaries whenever possible.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = (
    "Machine learning is a branch of artificial intelligence that focuses on "
    "building systems that learn from data. Unlike traditional programming where "
    "rules are explicitly coded, ML algorithms identify patterns in data and make "
    "decisions with minimal human intervention.\n\n"
    "Supervised learning is the most common paradigm. It uses labeled examples — "
    "pairs of inputs and desired outputs — to train a model that can predict outputs "
    "for new, unseen inputs. Classification (predicting categories) and regression "
    "(predicting continuous values) are the two main tasks.\n\n"
    "Unsupervised learning works with unlabeled data. The algorithm tries to discover "
    "hidden structure: clustering groups similar data points, while dimensionality "
    "reduction compresses data while preserving important relationships.\n\n"
    "Deep learning uses neural networks with many layers to learn hierarchical "
    "representations. Convolutional networks excel at images; recurrent networks "
    "handle sequences like text and time series. Transformers — the architecture "
    "behind GPT and BERT — have revolutionized natural language processing."
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(long_text)

print(f"Input length : {len(long_text)} chars")
print(f"Output chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

Notice how the splitter first tries double-newlines (paragraph boundaries), then single newlines, then sentence-ending periods, and so on. The `chunk_overlap` parameter keeps some trailing context from the previous chunk so that information straddling a boundary isn't lost.

**Rule of thumb for embedding models:** aim for 500–1000 characters per chunk (roughly 100–200 tokens). Set overlap to 10–20 % of chunk size.

## 1.2 — MarkdownHeaderTextSplitter

Documentation and README files are structured with headers. `MarkdownHeaderTextSplitter` splits at header boundaries **and** records the header hierarchy as metadata on each resulting `Document`. This metadata is gold for RAG — you can filter by section later.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_doc = """# Machine Learning Guide

This guide covers ML fundamentals.

## Supervised Learning

Supervised learning uses labeled data to train models.

### Classification

Classification predicts discrete categories like spam/not-spam.

### Regression

Regression predicts continuous values like house prices.

## Unsupervised Learning

Unsupervised learning finds patterns in unlabeled data.

### Clustering

Clustering groups similar data points together.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_docs = md_splitter.split_text(markdown_doc)

print(f"Generated {len(md_docs)} documents with header metadata:\n")
for i, doc in enumerate(md_docs):
    preview = doc.page_content[:80].replace("\n", "\\n")
    print(f"  Doc {i+1}:")
    print(f"    Content : {preview}...")
    print(f"    Metadata: {doc.metadata}")
    print()

Each section becomes its own `Document` and "knows" where it sits in the header hierarchy. When we upsert these into Pinecone later, we can attach this hierarchy as metadata and filter queries to a specific section.

## 1.3 — Code-Aware Splitting

`RecursiveCharacterTextSplitter.from_language()` understands the syntax of Python, JavaScript, and 15+ other languages. It splits at function/class boundaries instead of in the middle of a block.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

python_code = '''
def hello_world():
    print("Hello, World!")

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = None

    def load_data(self, path):
        with open(path) as f:
            self.data = f.read()
        return self.data

    def clean_data(self):
        if self.data is None:
            raise ValueError("No data loaded")
        return self.data.strip()

def train_model(data, epochs=10):
    for epoch in range(epochs):
        pass
    return "trained_model"
'''

py_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=200,
    chunk_overlap=20,
)

code_chunks = py_splitter.create_documents([python_code])

print(f"Generated {len(code_chunks)} code chunks:\n")
for i, doc in enumerate(code_chunks):
    has_class = "class " in doc.page_content
    has_def = "def " in doc.page_content
    kind = "CLASS" if has_class else ("FUNCTION" if has_def else "OTHER")
    preview = doc.page_content[:60].replace("\n", "\\n")
    print(f"  Chunk {i+1} [{kind:8}]: {preview}...")

print("\nPython separators LangChain uses:")
for sep in RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)[:6]:
    print(f"  {repr(sep)}")

## 1.4 — Sentence-Based (Semantic) Splitting

When you want chunks that always end at a sentence boundary, you can split on punctuation. This keeps each chunk semantically self-contained and avoids the "mid-thought" problem that character-based splitting sometimes creates.

In [ ]:
import re

def sentence_based_split(text, chunk_size=300, sentences_overlap=2):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = []
    current_len = 0

    for sentence in sentences:
        if current_len + len(sentence) > chunk_size and current:
            chunks.append(" ".join(current))
            current = current[-sentences_overlap:]
            current_len = sum(len(s) for s in current)
        current.append(sentence)
        current_len += len(sentence)

    if current:
        chunks.append(" ".join(current))
    return chunks

prose = (
    "Machine learning enables computers to learn from data. "
    "Unlike traditional programs, ML systems improve with experience. "
    "Supervised learning maps inputs to known outputs. "
    "Classification predicts categories while regression predicts numbers. "
    "Unsupervised learning discovers hidden patterns. "
    "Clustering is its most popular technique. "
    "Deep learning stacks many neural network layers. "
    "Transformers power modern NLP systems like GPT."
)

sentence_chunks = sentence_based_split(prose, chunk_size=200, sentences_overlap=1)

print(f"Sentence-based split produced {len(sentence_chunks)} chunks:\n")
for i, c in enumerate(sentence_chunks):
    print(f"  Chunk {i+1} ({len(c)} chars): {c}")
    print()

## 1.5 — Quick Reference: Which Splitter to Use

| Document Type | Recommended Splitter |
|---------------|---------------------|
| Plain text / prose | `RecursiveCharacterTextSplitter` (default separators) |
| Markdown docs | `MarkdownHeaderTextSplitter` (preserves header metadata) |
| Python code | `RecursiveCharacterTextSplitter.from_language(Language.PYTHON)` |
| JavaScript / TypeScript | `RecursiveCharacterTextSplitter.from_language(Language.JS)` |
| JSON / API responses | `RecursiveJsonSplitter` |
| HTML / web pages | `HTMLHeaderTextSplitter` |

**Chunk size recommendations:**
- For embeddings: 500–1000 characters (≈100–200 tokens)
- For LLM context: match the model's token limit minus your prompt
- Overlap: 10–20 % of chunk size

---

# Stage 2 — Document Loaders & Metadata

Real-world RAG systems rarely work with plain `.txt` files. You'll encounter Markdown docs, CSV FAQs, JSON API responses, and web pages. LangChain provides **100+ document loaders** with a common interface:

```python
loader = SomeLoader(...)
documents = loader.load()       # load all at once
for doc in loader.lazy_load():  # or stream lazily
    process(doc)
```

Every loader returns `Document` objects with:
- `page_content: str` — the text
- `metadata: dict` — source, page number, etc.

This means **your code stays the same** regardless of the input format.

## 2.1 — CSVLoader

Each CSV row becomes a `Document`, with column values in `page_content` and the row number in `metadata`. Perfect for FAQ datasets or product catalogs.

In [ ]:
import tempfile, os
from pathlib import Path
from langchain_community.document_loaders import CSVLoader

csv_content = """question,answer,category
What is machine learning?,ML is a subset of AI that learns from data,basics
How does gradient descent work?,It minimizes loss by following the gradient,optimization
What are neural networks?,Computational models inspired by the brain,deep learning
What is overfitting?,When a model memorizes training data too well,problems
"""

csv_dir = Path(tempfile.mkdtemp())
csv_path = csv_dir / "faq.csv"
csv_path.write_text(csv_content)

csv_loader = CSVLoader(file_path=str(csv_path))
csv_docs = csv_loader.load()

print(f"Loaded {len(csv_docs)} documents (one per row)\n")
for i, doc in enumerate(csv_docs[:3]):
    print(f"  Doc {i+1}:")
    print(f"    Content: {doc.page_content[:80]}...")
    print(f"    Row    : {doc.metadata.get('row', 'N/A')}")
    print()

## 2.2 — JSONLoader

`JSONLoader` uses **jq-style queries** to select which parts of a JSON structure become documents. You can also pull fields into metadata with a custom `metadata_func`.

In [ ]:
from langchain_community.document_loaders import JSONLoader

json_content = """[
    {"title": "Introduction to ML", "content": "Machine learning enables computers to learn from data without explicit programming.", "category": "basics"},
    {"title": "Neural Networks", "content": "Neural networks are composed of layers of interconnected nodes that process information.", "category": "deep-learning"},
    {"title": "Training Models", "content": "Model training involves feeding data through the network and adjusting weights.", "category": "training"}
]"""

json_dir = Path(tempfile.mkdtemp())
json_path = json_dir / "docs.json"
json_path.write_text(json_content)

json_loader = JSONLoader(
    file_path=str(json_path),
    jq_schema=".[]",
    content_key="content",
    metadata_func=lambda record, metadata: {
        **metadata,
        "title": record.get("title", ""),
        "category": record.get("category", ""),
    },
)

json_docs = json_loader.load()

print(f"Loaded {len(json_docs)} documents\n")
for i, doc in enumerate(json_docs):
    print(f"  Doc {i+1}:")
    print(f"    Content : {doc.page_content[:70]}...")
    print(f"    Title   : {doc.metadata.get('title')}")
    print(f"    Category: {doc.metadata.get('category')}")
    print()

## 2.3 — Metadata Schema Design

Metadata is what transforms a vector database from pure semantic search into a powerful **hybrid system** that combines meaning with structured attributes. Good metadata lets you answer questions like *"Find similar documents, but only from the last month, in the tutorials category."*

### Design Principles

1. **Start with your query patterns** — list every filter you'll need, then design fields for them.
2. **Keep it flat** — avoid nested structures; most vector DBs don't support them.
3. **Use the right types** — strings for categories, numbers for ranges, booleans for flags.
4. **Separate filtering fields from display fields** — not everything needs to be indexed.
5. **Always include source tracking** — essential for citations and debugging.

### Practical Schema

```python
metadata = {
    # Filterable
    "source": "ml_guide.pdf",
    "source_type": "pdf",
    "category": "tutorial",
    "year": 2024,

    # Positional
    "chunk_index": 0,
    "total_chunks": 10,

    # Display
    "section_heading": "Introduction to Neural Networks",
}
```

**Avoid:** `None`/`null` values (Pinecone ignores them), deeply nested objects, and arrays in filter fields.

## 2.4 — Upserting Documents with Metadata into Pinecone

Let's bring everything together: split documents, generate embeddings, build metadata, and upsert into a Pinecone index. We'll create a small knowledge base of AI/ML content that we'll query in Stage 3.

In [ ]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
embedder = SentenceTransformer("all-MiniLM-L6-v2")

INDEX_NAME = "w5-wednesday-demo"
DIMENSION = 384

if INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index '{INDEX_NAME}'")
else:
    print(f"Index '{INDEX_NAME}' already exists")

index = pc.Index(INDEX_NAME)
print(f"Index stats: {index.describe_index_stats()}")

In [ ]:
documents = [
    {
        "id": "ml_1",
        "content": "Machine learning enables computers to learn from data without being explicitly programmed.",
        "metadata": {"source": "ml_guide.pdf", "category": "tutorial", "year": 2024},
    },
    {
        "id": "ml_2",
        "content": "Supervised learning uses labeled data — pairs of inputs and desired outputs — to train predictive models.",
        "metadata": {"source": "ml_guide.pdf", "category": "tutorial", "year": 2024},
    },
    {
        "id": "dl_1",
        "content": "Deep learning uses neural networks with many layers to learn hierarchical representations of data.",
        "metadata": {"source": "deep_learning.md", "category": "reference", "year": 2024},
    },
    {
        "id": "nlp_1",
        "content": "Natural language processing enables computers to understand, interpret, and generate human language.",
        "metadata": {"source": "nlp_guide.html", "category": "tutorial", "year": 2023},
    },
    {
        "id": "rag_1",
        "content": "RAG combines retrieval from a vector database with LLM generation to produce grounded, factual answers.",
        "metadata": {"source": "rag_patterns.pdf", "category": "reference", "year": 2024},
    },
    {
        "id": "rag_2",
        "content": "In a RAG pipeline, the user query is embedded, similar documents are retrieved, and an LLM synthesizes the answer.",
        "metadata": {"source": "rag_patterns.pdf", "category": "reference", "year": 2024},
    },
    {
        "id": "embed_1",
        "content": "Embedding models convert text into dense vector representations that capture semantic meaning.",
        "metadata": {"source": "embeddings_101.md", "category": "tutorial", "year": 2024},
    },
    {
        "id": "hybrid_1",
        "content": "Hybrid search combines dense vector similarity with sparse keyword matching (BM25) for better precision on specific terms.",
        "metadata": {"source": "search_patterns.pdf", "category": "reference", "year": 2024},
    },
]

contents = [d["content"] for d in documents]
embeddings = embedder.encode(contents).tolist()

vectors_to_upsert = [
    (doc["id"], emb, doc["metadata"])
    for doc, emb in zip(documents, embeddings)
]

index.upsert(vectors=vectors_to_upsert)
print(f"Upserted {len(vectors_to_upsert)} vectors with metadata")
print(f"Index stats: {index.describe_index_stats()}")

We now have eight documents in Pinecone, each with `source`, `category`, and `year` metadata. Let's query them.

---

# Stage 3 — Semantic Search with Pinecone

With our vectors stored and metadata attached, it's time to run real queries. We'll cover:

1. **Basic similarity search** — finding the most semantically similar documents.
2. **Filtered search** — combining similarity with metadata constraints.
3. **Score interpretation** — understanding what cosine scores actually mean.
4. **Search limitations** — where vector search breaks down and what to do about it.

## 3.1 — Basic Similarity Search

In [ ]:
import time
time.sleep(2)

query = "How do machines learn from data?"
query_embedding = embedder.encode([query]).tolist()

results = index.query(
    vector=query_embedding[0],
    top_k=5,
    include_metadata=True,
)

print(f'Query: "{query}"\n')
print(f"Top {len(results.matches)} results (no filter):\n")
for match in results.matches:
    meta = match.metadata
    print(f"  [{match.score:.4f}] (id={match.id})")
    print(f"           source={meta.get('source')}, category={meta.get('category')}")

Pinecone returns results ranked by **cosine similarity** (higher is more similar). The `score` field ranges from 0 to 1 for cosine:

| Score Range | Interpretation |
|-------------|---------------|
| 0.8 – 1.0 | Very similar — strong semantic match |
| 0.6 – 0.8 | Related — the topic overlaps |
| 0.3 – 0.6 | Loosely related — some shared concepts |
| < 0.3 | Mostly unrelated |

## 3.2 — Filtered Search with Metadata

Pure semantic search returns the most similar vectors, period. But real applications need more control:

- *"Find similar documents, but only tutorials."*
- *"Show me results from 2024 only."*
- *"Search within a specific source file."*

Pinecone supports a `filter` parameter with comparison and logical operators.

### Pinecone Filter Operators

| Operator | Meaning | Example |
|----------|---------|--------|
| `$eq` | Equal | `{"category": {"$eq": "tutorial"}}` |
| `$ne` | Not equal | `{"category": {"$ne": "draft"}}` |
| `$gt` | Greater than | `{"year": {"$gt": 2023}}` |
| `$gte` | Greater or equal | `{"year": {"$gte": 2024}}` |
| `$lt` | Less than | `{"year": {"$lt": 2025}}` |
| `$lte` | Less or equal | `{"year": {"$lte": 2024}}` |
| `$in` | In list | `{"category": {"$in": ["tutorial", "guide"]}}` |
| `$nin` | Not in list | `{"category": {"$nin": ["draft"]}}` |
| `$and` | All conditions match | `{"$and": [{...}, {...}]}` |
| `$or` | Any condition matches | `{"$or": [{...}, {...}]}` |

In [ ]:
query = "How do machines learn from data?"
query_emb = embedder.encode([query]).tolist()[0]

print(f'Query: "{query}"')
print("=" * 60)

# Filter 1: category = "tutorial"
print("\n[Filter] category == 'tutorial'")
results = index.query(
    vector=query_emb,
    top_k=5,
    include_metadata=True,
    filter={"category": {"$eq": "tutorial"}},
)
for m in results.matches:
    print(f"  [{m.score:.4f}] {m.id} — {m.metadata.get('source')}")

# Filter 2: year > 2023
print("\n[Filter] year > 2023")
results = index.query(
    vector=query_emb,
    top_k=5,
    include_metadata=True,
    filter={"year": {"$gt": 2023}},
)
for m in results.matches:
    print(f"  [{m.score:.4f}] {m.id} — year={m.metadata.get('year')}")

# Filter 3: AND logic — tutorial AND 2024
print("\n[Filter] category == 'tutorial' AND year == 2024")
results = index.query(
    vector=query_emb,
    top_k=5,
    include_metadata=True,
    filter={
        "$and": [
            {"category": {"$eq": "tutorial"}},
            {"year": {"$eq": 2024}},
        ]
    },
)
for m in results.matches:
    print(f"  [{m.score:.4f}] {m.id} — {m.metadata.get('category')}, {m.metadata.get('year')}")

In [ ]:
# Filter 4: OR logic — source is one of two PDFs
print('[Filter] source IN ["ml_guide.pdf", "rag_patterns.pdf"]')
results = index.query(
    vector=query_emb,
    top_k=5,
    include_metadata=True,
    filter={"source": {"$in": ["ml_guide.pdf", "rag_patterns.pdf"]}},
)
for m in results.matches:
    print(f"  [{m.score:.4f}] {m.id} — {m.metadata.get('source')}")

# Filter 5: Nested AND/OR
print('\n[Filter] year == 2024 AND (category == "tutorial" OR category == "reference")')
results = index.query(
    vector=query_emb,
    top_k=5,
    include_metadata=True,
    filter={
        "$and": [
            {"year": {"$eq": 2024}},
            {"$or": [
                {"category": {"$eq": "tutorial"}},
                {"category": {"$eq": "reference"}},
            ]},
        ]
    },
)
for m in results.matches:
    print(f"  [{m.score:.4f}] {m.id} — {m.metadata.get('category')}, {m.metadata.get('year')}")

## 3.3 — Score Interpretation & Thresholds

Pinecone returns **cosine similarity** (not distance). Higher is better. But what counts as "relevant" depends on your use case:

| Use Case | Suggested Threshold | Why |
|----------|--------------------|---------|
| Semantic search (user-facing) | score ≥ 0.5 | Users expect relevant results |
| FAQ matching | score ≥ 0.7 | Need a close match to the original question |
| Deduplication | score ≥ 0.9 | Content must be near-identical |
| RAG context retrieval | Take top-K, no hard cutoff | Let the LLM judge relevance |

**Practical advice:** analyze your score distribution before picking a threshold. The right cutoff depends on your data and embedding model.

In [ ]:
def search_with_threshold(index, embedder, query, threshold=0.5, top_k=5):
    query_emb = embedder.encode([query]).tolist()[0]
    results = index.query(
        vector=query_emb,
        top_k=top_k,
        include_metadata=True,
    )

    relevant = [m for m in results.matches if m.score >= threshold]

    if not relevant:
        return {"status": "no_relevant_results", "results": []}
    return {
        "status": "success",
        "results": [
            {"id": m.id, "score": m.score, "metadata": m.metadata}
            for m in relevant
        ],
    }


def score_to_label(score):
    if score >= 0.8:
        return "HIGH"
    elif score >= 0.6:
        return "MEDIUM"
    return "LOW"


print("--- Threshold demo ---\n")
result = search_with_threshold(index, embedder, "How does machine learning work?", threshold=0.5)
print(f"Status: {result['status']}")
for r in result["results"]:
    print(f"  [{score_to_label(r['score']):6}] {r['score']:.4f} — {r['id']} ({r['metadata'].get('source')})")

print("\n--- Strict threshold (0.8) ---\n")
result = search_with_threshold(index, embedder, "How does machine learning work?", threshold=0.8)
print(f"Status: {result['status']}, matches: {len(result['results'])}")

## 3.4 — Vector Search Limitations

Vector search is powerful, but it has blind spots you need to recognize:

| Limitation | Example | Mitigation |
|------------|---------|------------|
| **Keyword specificity** | Query: `"Error code XJ7742"` — vectors dilute specific identifiers | Add keyword (BM25) search alongside vectors |
| **Negation / logic** | Query: `"Documents NOT about ML"` — vectors can't negate | Use metadata filters to exclude categories |
| **Recency blindness** | Query: `"latest news"` — vectors have no concept of time | Attach `date` metadata and filter |
| **Out-of-domain queries** | Querying a medical DB with cooking questions | Use domain-specific embedding models |
| **Ambiguity** | `"Python"` — programming language or snake? | Larger models + domain-specific data help |

The key takeaway: **hybrid search** (vector similarity + keyword matching + metadata filtering) outperforms any single approach in production.

In [ ]:
print("=== Demonstrating a limitation: keyword specificity ===")
print()

keyword_query = "Error code XJ7742"
keyword_emb = embedder.encode([keyword_query]).tolist()[0]

results = index.query(vector=keyword_emb, top_k=3, include_metadata=True)

print(f'Query: "{keyword_query}"')
print(f"Top results (vector search only):\n")
for m in results.matches:
    print(f"  [{m.score:.4f}] {m.id} — {m.metadata.get('source')}")

print()
print("Observation: vector search returns something, but none of our")
print("documents actually mention that error code. The scores are low,")
print("confirming a weak match. A keyword search would correctly return")
print("zero results. This is why hybrid search exists.")

## 3.5 — Hybrid Search Concepts

Hybrid search combines **dense retrieval** (vector similarity) with **sparse retrieval** (keyword/BM25 matching). This gives you the best of both worlds:

- **Dense (vectors):** great at understanding *meaning* — "How do machines learn?" matches "ML algorithms improve with experience."
- **Sparse (keywords):** great at exact matches — "XJ7742" only matches documents containing that exact string.

Pinecone supports hybrid search natively through **sparse-dense vectors**. The typical workflow is:

1. Generate a **dense embedding** (e.g., from `all-MiniLM-L6-v2`).
2. Generate a **sparse representation** (e.g., BM25 or SPLADE).
3. Upsert both into Pinecone.
4. At query time, provide both and let Pinecone combine the scores.

We'll see this in practice when we build end-to-end RAG pipelines on Thursday.

## 3.6 — Complete Search Pipeline Example

Let's wrap everything into a reusable search function that demonstrates the full query lifecycle: embed the query, search with optional filters, interpret scores, and format results.

In [ ]:
def semantic_search(
    index,
    embedder,
    query,
    top_k=5,
    filter_dict=None,
    threshold=None,
):
    query_emb = embedder.encode([query]).tolist()[0]

    results = index.query(
        vector=query_emb,
        top_k=top_k,
        include_metadata=True,
        filter=filter_dict,
    )

    formatted = []
    for m in results.matches:
        if threshold and m.score < threshold:
            continue
        confidence = "HIGH" if m.score >= 0.8 else ("MEDIUM" if m.score >= 0.6 else "LOW")
        formatted.append({
            "id": m.id,
            "score": round(m.score, 4),
            "confidence": confidence,
            "metadata": m.metadata,
        })
    return formatted


print("=" * 60)
print("Query 1: General search")
print("=" * 60)
for r in semantic_search(index, embedder, "What is retrieval-augmented generation?"):
    print(f"  [{r['confidence']:6}] {r['score']} — {r['id']} ({r['metadata'].get('source')})")

print()
print("=" * 60)
print("Query 2: Tutorials only, 2024+")
print("=" * 60)
for r in semantic_search(
    index, embedder,
    "How do neural networks learn?",
    filter_dict={"$and": [{"category": {"$eq": "tutorial"}}, {"year": {"$gte": 2024}}]},
):
    print(f"  [{r['confidence']:6}] {r['score']} — {r['id']} ({r['metadata'].get('category')})")

print()
print("=" * 60)
print("Query 3: References about RAG")
print("=" * 60)
for r in semantic_search(
    index, embedder,
    "retrieval augmented generation pipeline",
    filter_dict={"category": {"$eq": "reference"}},
    threshold=0.4,
):
    print(f"  [{r['confidence']:6}] {r['score']} — {r['id']} ({r['metadata'].get('source')})")

## Cleanup (Optional)

Uncomment the cell below to delete the demo index when you're done.

In [ ]:
# pc.delete_index(INDEX_NAME)
# print(f"Deleted index '{INDEX_NAME}'")

---

# Key Takeaways

1. **Use the right splitter for the job.** `RecursiveCharacterTextSplitter` is the default workhorse; use `MarkdownHeaderTextSplitter` for structured docs and `from_language()` for code. Aim for 500–1000 character chunks with 10–20 % overlap.

2. **LangChain document loaders** give you a uniform `Document(page_content, metadata)` interface across CSV, JSON, PDF, HTML, and dozens more formats. Your downstream code stays the same.

3. **Metadata is what makes search precise.** Always attach `source`, `category`, and temporal fields. Keep schemas flat, avoid nulls, and design fields around your expected query patterns.

4. **Pinecone filters are powerful.** Use `$eq`, `$gt`, `$in`, `$and`, `$or` to combine semantic similarity with structured constraints — this is how production RAG systems deliver focused results.

5. **Interpret scores carefully.** Cosine similarity > 0.8 is a strong match; < 0.5 is often noise. Analyze your score distribution before choosing thresholds.

6. **Know the limits.** Vector search struggles with exact keywords, negation, and recency. Hybrid search (dense + sparse) and metadata filters are your mitigation tools.

---

## Looking Ahead — Thursday

Tomorrow we bring everything together into **end-to-end RAG pipelines**: ingesting real documents, chunking them with the splitters we learned today, storing them in Pinecone with rich metadata, retrieving context with filtered search, and generating grounded answers with an LLM. The full cycle from document to answer.